In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from scipy.stats import zscore
from src import Questions
import matplotlib.patches as mpatches
from functools import reduce

plt.rcParams.update({'font.size': 17})

In [10]:
import pandas as pd
import src.utils.paper_artefacts as pa

In [12]:
# NAIVE P VALUES
REJECT_THRESHOLD = 0.05
# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 0
task_lingcat_scores_by_model = {}
for mmm in pa.ALL_TASKTYPE_DF_BY_MODEL.keys():
    task_lingcat_scores_by_model[mmm] = {}
    for tt in pa.TASK_TYPES:
        task_lingcat_scores_by_model[mmm][tt] = {}
        taskdf = pa.ALL_TASKTYPE_DF_BY_MODEL[mmm][tt].drop(columns=['id'])
        cols = list(taskdf.columns)
        for i in range(0, len(cols), 4):
            chunk = cols[i:i+4]
            task_qidx = chunk[-1].split("-")[-2]
            if not any(task_qidx.startswith(prefix) for prefix in pa.EXCLUDED_SCORES):
                task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                chunkdf = taskdf[chunk].dropna()
                for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()

rejectable_objlist =[]
for modelname in task_lingcat_scores_by_model.keys():
    row_instance = {"model" : modelname}
    for tg in task_lingcat_scores_by_model[modelname].keys():
        unfair_tasks = []
        for qidx in task_lingcat_scores_by_model[modelname][tg].keys():
            question_id = int(qidx.split("_")[0])
            clip_llm_judge = True if question_id in pa.LLM_JUDGE_TASKS else False
            qid_min = pa.INDEX_MM_LOOKUP_TABLE[question_id]["min"]
            qid_max = pa.INDEX_MM_LOOKUP_TABLE[question_id]["max"]

            this_task_unfair = False
            ttest_result = pa.compute_pairwise_t_test(
                task_lingcat_scores_by_model[modelname][tg][qidx],
                qid_min,
                qid_max,
                clip_llm_judge
            )
            for ttest_value in ttest_result.values():
                if ttest_value < REJECT_THRESHOLD:
                    this_task_unfair = True
            unfair_tasks.append(this_task_unfair)
        row_instance[tg] = sum(unfair_tasks)/len(unfair_tasks) 
    rejectable_objlist.append(row_instance)
rejectable_df = pd.DataFrame(rejectable_objlist)
sortdcols = ["model"] + sorted([c for c in rejectable_df.columns if c != "model"])
rejectable_df = rejectable_df[sortdcols]

if CHAR_COLUMN == 0:
    rejectable_df.to_csv(f"score_tables/ptone_ptt.csv")
else:
    rejectable_df.to_csv(f"score_tables/pgrammar_ptt.csv")

/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_stats_py.py:1114: RuntimeWarning: divide by zero encountered in divide
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_stats_py.py:1114: RuntimeWarning: invalid value encountered in scalar multiply
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [13]:
print(task_lingcat_scores_by_model.keys())
print(task_lingcat_scores_by_model['gpt_4o'].keys())
print(task_lingcat_scores_by_model['gpt_4o']['syntaxes'].keys())
print(task_lingcat_scores_by_model['gpt_4o']['syntaxes']['16_0'].keys())

dict_keys(['gpt_4o', 'llama31_70b_Inst', 'llama31_8B_Inst', 'llama32_1B_Inst', 'qwen25_72b_Inst', 'Qwen25_7B_Inst', 'Qwen25_1dot5B_Inst'])
dict_keys(['summarization', 'qa_extraction', 'conditioned_text_gen', 'knowledge_and_reasoning', 'machine_translation', 'semantics', 'syntaxes'])
dict_keys(['16_0', '17_0', '26_0', '26_1'])
dict_keys(['declarative', 'imperative', 'interrogative'])


In [16]:
# ANOVA WITH BONFERRONI
REJECT_THRESHOLD = 0.05

# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 0
task_lingcat_scores_by_model = {}
for mmm in pa.ALL_TASKTYPE_DF_BY_MODEL.keys():
    task_lingcat_scores_by_model[mmm] = {}
    for tt in pa.TASK_TYPES:
        task_lingcat_scores_by_model[mmm][tt] = {}
        taskdf = pa.ALL_TASKTYPE_DF_BY_MODEL[mmm][tt].drop(columns=['id'])
        cols = list(taskdf.columns)
        for i in range(0, len(cols), 4):
            chunk = cols[i:i+4]
            task_qidx = chunk[-1].split("-")[-2]
            if not any(task_qidx.startswith(prefix) for prefix in pa.EXCLUDED_SCORES):
                task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                chunkdf = taskdf[chunk].dropna()
                for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()

for modl in task_lingcat_scores_by_model.keys():
    for tg in task_lingcat_scores_by_model[modl].keys():
        task_count_in_tg = len(task_lingcat_scores_by_model[modl][tg].keys())
        for qdi in task_lingcat_scores_by_model[modl][tg].keys():
            question_id = int(qdi.split("_")[0])
            clip_llm_judge = True if question_id in pa.LLM_JUDGE_TASKS else False
            qid_min = pa.INDEX_MM_LOOKUP_TABLE[question_id]["min"]
            qid_max = pa.INDEX_MM_LOOKUP_TABLE[question_id]["max"]
            f, p = pa.compute_anova_score(task_lingcat_scores_by_model[modl][tg][qdi], qid_min, qid_max, clip_llm_judge)
            adjusted_p = p * task_count_in_tg * 7
            if (adjusted_p < REJECT_THRESHOLD):
                print(modl, "REJECTED!", f"task_group-{tg}", f"task-{qdi}", adjusted_p)

gpt_4o REJECTED! task_group-machine_translation task-2_3 5.444476652227586e-05
gpt_4o REJECTED! task_group-machine_translation task-2_5 0.0011074897048645085
gpt_4o REJECTED! task_group-machine_translation task-2_2 8.862490932173876e-09
gpt_4o REJECTED! task_group-machine_translation task-2_1 0.0001371644479367351
gpt_4o REJECTED! task_group-machine_translation task-2_6 2.5642360800692208e-06
gpt_4o REJECTED! task_group-machine_translation task-2_8 1.715388367342287e-09
gpt_4o REJECTED! task_group-machine_translation task-2_9 1.6767929786821524e-06
gpt_4o REJECTED! task_group-machine_translation task-2_7 0.0002774259635318474
llama31_8B_Inst REJECTED! task_group-machine_translation task-2_6 0.00013221625740463994
llama32_1B_Inst REJECTED! task_group-machine_translation task-2_11 0.035036966595370235
llama32_1B_Inst REJECTED! task_group-machine_translation task-2_5 0.038850884911009756
llama32_1B_Inst REJECTED! task_group-machine_translation task-2_7 0.0012878002935286772
qwen25_72b_Ins

/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)
